# 06b · Sensitivity analyses and outcome components

Runs every sensitivity analysis in Table S3 and the exploratory outcome components and heart-failure checks in Table S8.

**Reads (`DATA_DIR`):** `analysis_frame`, `mi`, `landmark_build`, `covariates`, `covariates_preindex`, `covariates_oldwindow`, `quarterly_unique`, the fitted model of the selected solution `gbtm/k<K>.rds`, `trajectory_classes_k5` (alternative solution); and `results/06_main.json`, `results/spline_knots.json`, `results/05_trajectories.json`

**Produces (aggregate, `results/`):** `tableS3.csv`, `tableS8a_components.csv`, `tableS8b_heart_failure.csv`, `figureS3.csv`, `06b_sensitivity.json`

**Paper items:** Table S3; Table S8; Figure S3; censoring results in the text (odds ratios for early censoring, weight distribution, effective sample size).

**Runtime:** roughly 1.5–3 hours. Run notebook 06 first.

## 0 · Setup

In [ ]:
# ---- edit these ----
REPO_DIR   = "/content/drive/MyDrive/Ronveer Sandhu/Project 1 -MIMIC analysis-hypertension retention/htn-bp-followup-mimic"
MIMIC_HOSP = "/content/drive/MyDrive/MIMIC Data/mimic-iv-2.2/hosp"
MIMIC_ED   = "/content/drive/MyDrive/MIMIC Data/mimic-iv-2.2/ed"
DATA_DIR   = "/content/drive/MyDrive/htn_bp_data"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!pip -q install duckdb==1.3.2 lifelines==0.30.3 scikit-learn==1.6.1 statsmodels==0.15.0

import os, sys, json, re, time, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
from htn_utils import *
from landmark_models import *

import numpy as np
import pandas as pd
import statsmodels.api as sm
from lifelines import CoxPHFitter

RESULTS_DIR = os.path.join(REPO_DIR, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

A   = load_df(DATA_DIR, "analysis_frame").reset_index(drop=True)
MI  = load_df(DATA_DIR, "mi")
lb  = load_df(DATA_DIR, "landmark_build")
cov = load_df(DATA_DIR, "covariates")
pre = load_df(DATA_DIR, "covariates_preindex")
ow  = load_df(DATA_DIR, "covariates_oldwindow")
qu  = load_df(DATA_DIR, "quarterly_unique")
KN  = json.load(open(os.path.join(RESULTS_DIR, "spline_knots.json")))
OUTR = json.load(open(os.path.join(RESULTS_DIR, "06_main.json")))
TRAJ = json.load(open(os.path.join(RESULTS_DIR, "05_trajectories.json")))

M  = int(MI[".imp"].max())
assert (MI[MI[".imp"] == 1].subject_id.values == A.subject_id.values).all()
T  = A.time.astype(float).values
E1 = (A.status == 1).astype(int).values
set_class_order(TRAJ["class_order"]); SM = key_class(); K = int(TRAJ["k_selected"])
Cl = class_dummies(A); G1 = A[["gap365"]].astype(float)
assert set(Cl.columns) == set(ORDER[1:]), "class dummies do not match the selected classes"
print(f"{K} trajectory classes:", ORDER, "| Table S3 class contrast:", SM)

def imputed(m):
    F = A.copy(); F[MISS] = MI.loc[MI[".imp"] == m, MISS].values; return F

def fit_gen(F, expo, extra=None, T_=T, E_=E1, linear=False):
    parts = [pd.DataFrame({"T": T_, "E": E_}), build_X(F, KN, linear=linear)] + ([extra] if extra is not None else []) + [expo]
    d = pd.concat(parts, axis=1); d = d.loc[:, d.std() > 0]
    c = CoxPHFitter(penalizer=1e-6).fit(d, "T", "E")
    k = [x for x in expo.columns if x in d.columns]
    return c.params_[k].values, c.variance_matrix_.loc[k, k].values, k

ALL = lambda fn: pool([fn(imputed(m)) for m in range(1, M + 1)])
print(f"n={len(A):,} | events={E1.sum()} | imputations={M}")

## 1 · Table S3: estimators, gap and class definitions, covariates and missing data

Each row reports the hazard ratio for a gap longer than 365 days, for the key class contrast (Sustained Contact vs Minimal Contact with 4 classes; Sustained Moderate Contact with 5), and the overall test for class. The alternative trajectory solution is the last row of this section.

In [ ]:
rows = []
def record(name, n, ev, g=None, c=None, gap_label="gap365", key=None):
    key = key or SM
    r = {"analysis": name, "n": n, "events": ev, "class_contrast": key if c is not None else ""}
    if g is not None:
        gt = g[0]; r.update(gap_HR=gt.HR.iloc[0], gap_lo=gt.lo.iloc[0], gap_hi=gt.hi.iloc[0])
    if c is not None:
        ct, _, cp = c
        if key in ct.index: r.update(sm_HR=ct.loc[key, "HR"], sm_lo=ct.loc[key, "lo"], sm_hi=ct.loc[key, "hi"])
        r["class_p"] = cp
    rows.append(r)
    print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in r.items()})

def from_json(key):
    t = pd.DataFrame(OUTR[key]["table"]).T
    return t, OUTR[key].get("wald"), OUTR[key]["p_overall"]

t0 = time.time(); n, ev = len(A), int(E1.sum())
# estimators (from notebook 06)
record("Primary: cause-specific Cox", n, ev, from_json("gap365"), from_json("class"))
record("Fine-Gray", n, ev, from_json("fg_gap365"), from_json("fg_class"))
record("Doubly robust", n, ev, from_json("dr_gap365"), from_json("dr_class"))
cj = from_json("class+gap365")
record("Class and gap in the same model", n, ev, (cj[0].loc[["gap365"]],), (cj[0].drop(index="gap365"), None, OUTR["class_given_gap_p"]))

# gap definitions
for cut in [180, 270]:
    t_ = pd.DataFrame(OUTR["gap_cutoffs"]).T.loc[[f"gap_gt{cut}"]]
    record(f"Gap >{cut} days", n, ev, (t_,))
record("Gap >365 days within Minimal Contact", int((A.label == "Minimal Contact").sum()),
       int(E1[A.label == "Minimal Contact"].sum()), from_json("gap365_in_Minimal"))

# class definitions
Cp = A[[f"p_{c}" for c in ORDER[1:]]].copy(); Cp.columns = ORDER[1:]
record("Posterior probabilities in place of class assignment", n, ev, None, ALL(lambda F: fit_gen(F, Cp)))

# alternative trajectory solution (5 classes when 4 are selected)
ALT = TRAJ["alternative"]
ca = load_df(DATA_DIR, f"trajectory_classes_k{ALT['k']}").set_index("subject_id").loc[A.subject_id, "label"].values
Ca = (pd.get_dummies(pd.Categorical(ca, categories=ALT["class_order"]), dtype=float)
        .drop(columns="Minimal Contact").set_index(A.index))
akey = "Sustained Moderate Contact" if "Sustained Moderate Contact" in ALT["class_order"] else "Sustained Contact"
alt_res = ALL(lambda F: fit_gen(F, Ca))
record(f"{ALT['k']}-class trajectory solution", n, ev, None, alt_res, key=akey)
print("   alternative solution, all classes:", alt_res[0][["HR", "lo", "hi"]].round(2).to_dict("index"))
print(f"[{(time.time()-t0)/60:.1f} min]")

In [ ]:
# covariates and missing data
t0 = time.time()
Fo = A.copy()
owi = ow.set_index("subject_id")
for c in ["sbp_baseline", "dbp_baseline", "egfr", "bmi", "hba1c"]:
    Fo[c] = owi.loc[A.subject_id, c].values
Fo, indo = with_indicators(Fo)
record("Covariates averaged from -365 to +90 days, missing indicators", n, ev,
       single(fit_gen(Fo, G1, indo)), single(fit_gen(Fo, Cl, indo)))
record("Linear covariates (no splines)", n, ev,
       ALL(lambda F: fit_gen(F, G1, linear=True)), ALL(lambda F: fit_gen(F, Cl, linear=True)))
Fi, ind = with_indicators(A)
record("Missing-indicator method", n, ev, single(fit_gen(Fi, G1, ind)), single(fit_gen(Fi, Cl, ind)))
cc = A[MISS].notna().all(1)
Fc = A[cc].reset_index(drop=True); Tc = Fc.time.astype(float).values; Ec = (Fc.status == 1).astype(int).values
record("Complete cases", int(cc.sum()), int(Ec.sum()),
       single(fit_gen(Fc, Fc[["gap365"]].astype(float), T_=Tc, E_=Ec)),
       single(fit_gen(Fc, class_dummies(Fc), T_=Tc, E_=Ec)))

mask = {c: A[c].isna().values for c in MISS}
for lab, (c, dlt) in {"HbA1c +0.5": ("hba1c", .5), "HbA1c +1.0": ("hba1c", 1.), "BMI +3": ("bmi", 3.),
                      "BMI -3": ("bmi", -3.), "eGFR -10": ("egfr", -10.)}.items():
    def shifted(m, c=c, dlt=dlt):
        F = imputed(m); F.loc[mask[c], c] += dlt; return F
    record(f"Delta-adjusted imputation: {lab}", n, ev,
           pool([fit_gen(shifted(m), G1) for m in range(1, M + 1)]),
           pool([fit_gen(shifted(m), Cl) for m in range(1, M + 1)]))
print(f"[{(time.time()-t0)/60:.1f} min]")

## 2 · Table S3: other populations

The day-730-information sample and the sample keeping patients with events before index include patients outside the primary population. They are assigned trajectory classes from the fitted model of the selected solution **without refitting** (posterior probabilities computed from the model's parameters), use pre-index covariates with the missing-indicator method, and use the same follow-up rules.

In [ ]:
%load_ext rpy2.ipython
GBTM_DIR = os.path.join(DATA_DIR, "gbtm")
RDS5 = os.path.join(GBTM_DIR, f"k{K}.rds")
PAR_CSV = os.path.join(GBTM_DIR, f"k{K}_best.csv")

In [ ]:
%%R -i RDS5 -i PAR_CSV
m <- readRDS(RDS5); b <- m$best
write.csv(data.frame(name = names(b), value = as.numeric(b)), PAR_CSV, row.names = FALSE)

In [ ]:
from scipy.stats import norm as N
v = pd.read_csv(PAR_CSV).value.values; G = K
xi = np.append(v[:G - 1], 0.0); sig = v[-1]; fx = v[G - 1:-1]       # G-1 membership, 3G fixed, 1 residual SD
assert len(fx) == 3 * G
NAMES = {int(k): lab for k, lab in TRAJ["class_numbers_to_names"].items()}

def ymat(ids):
    g = pd.MultiIndex.from_product([ids, range(1, 8)], names=["subject_id", "q"]).to_frame(index=False)
    g = g.merge(qu, on=["subject_id", "q"], how="left").fillna({"n_unique": 0})
    return np.log1p(g.n_unique.values.reshape(len(ids), 7))

def posterior(Y, order):
    q = np.arange(1, 8)
    B = fx.reshape(3, G).T if order == "var" else fx.reshape(G, 3)
    mu = B[:, [0]] + B[:, [1]] * q + B[:, [2]] * q ** 2
    ll = np.stack([N.logpdf(Y, mu[g], sig).sum(1) for g in range(G)], 1) + np.log(np.exp(xi) / np.exp(xi).sum())
    ll -= ll.max(1, keepdims=True); P = np.exp(ll); return P / P.sum(1, keepdims=True)

# the parameter layout is checked against the classes the model assigned in notebook 05
Yv = ymat(A.subject_id.values)
assigned = A.label.map({lab: k for k, lab in NAMES.items()}).values
agree_ = {o: (posterior(Yv, o).argmax(1) + 1 == assigned).mean() for o in ["var", "class"]}
ORD = max(agree_, key=agree_.get)
print("agreement with assigned classes by parameter layout:", {k: round(x, 4) for k, x in agree_.items()})
assert agree_[ORD] > 0.995, "posterior rebuild failed"

In [ ]:
prei = pre.set_index("subject_id")
def make_frame(p):
    F = (p[["subject_id", "mace_day", "death_day", "fu_end", "longest_gap", "gapped", "age_index",
            "idx_adm_type", "idx_los", "idx_ed", "idx_disch", "has_idx_adm"]]
           .merge(cov[["subject_id", "gender", "race", "insurance", "n_ed_prior", "n_inp_prior", "n_enc_prior",
                       "n_antihtn_drugs", "elixhauser_vw"]], on="subject_id").reset_index(drop=True))
    cols = ["sbp_baseline", "dbp_baseline", "egfr", "bmi", "hba1c"] + (["n_antihtn_drugs"] if "n_antihtn_drugs" in prei else [])
    if "n_antihtn_drugs" in prei: F = F.drop(columns="n_antihtn_drugs")
    F = F.join(prei[cols], on="subject_id")
    F["ins_grp"] = np.where(F.insurance.eq("Medicaid"), "Medicaid", np.where(F.insurance.eq("Medicare"), "Medicare", "Other"))
    for s, c in [("log_enc_prior", "n_enc_prior"), ("log_ed_prior", "n_ed_prior"),
                 ("log_inp_prior", "n_inp_prior"), ("log_idx_los", "idx_los")]:
        F[s] = np.log1p(F[c].clip(lower=0))
    F["sbp_baseline"] = F.sbp_baseline.fillna(F.sbp_baseline.median())
    F["dbp_baseline"] = F.dbp_baseline.fillna(F.dbp_baseline.median())
    F["label"] = pd.Series(posterior(ymat(F.subject_id.values), ORD).argmax(1) + 1).map(NAMES).values
    F["gap365"] = F.gapped.astype(float); F["gap_days"] = F.longest_gap.astype(float)
    ev_ = F.mace_day.notna() & (F.mace_day > 730) & (F.mace_day <= F.fu_end)
    dt_ = ~ev_ & F.death_day.notna() & (F.death_day > 730) & (F.death_day <= F.fu_end)
    F["T"] = (np.where(ev_, F.mace_day, np.where(dt_, F.death_day, F.fu_end)) - 730).astype(float)
    F["E"] = ev_.astype(int)
    return F

for name, p in [("Survival confirmed with day-730 information only", lb[lb.in_day730_info]),
                ("Events before index kept", lb[lb.in_landmark_prior_ok])]:
    F = make_frame(p); Fi_, ind_ = with_indicators(F)
    record(name, len(F), int(F.E.sum()),
           single(fit_gen(Fi_, F[["gap365"]], ind_, T_=F["T"].values, E_=F.E.values)),
           single(fit_gen(Fi_, class_dummies(F), ind_, T_=F["T"].values, E_=F.E.values)))
    print("   classes:", F.label.value_counts().reindex(ORDER).to_dict())

## 3 · Table S3: censoring

**Censoring at the end of database coverage only:** follow-up ends at the end of coverage, day 1,460, or death (no death-capture window).

**Censoring weights:** each patient is weighted by the inverse probability of remaining under follow-up, from pooled logistic models for censoring in eight 91-day intervals (denominator: interval, covariates, both contact measures; numerator: interval and contact measures), multiplied over earlier intervals, stabilized, truncated at the 1st/99th or 5th/95th percentiles, refitted in each imputation; weighted cause-specific Cox models (R `survival`, robust standard errors).

In [ ]:
t0 = time.time()
end = np.minimum(A.admin_day.values, 1460)
ev_ = A.mace_day.notna() & (A.mace_day > 730) & (A.mace_day <= end)
dt_ = ~ev_ & A.death_day.notna() & (A.death_day > 730) & (A.death_day <= end)
T7 = (np.where(ev_, A.mace_day, np.where(dt_, A.death_day, end)) - 730).astype(float); E7 = ev_.astype(int).values
record("Censoring at the end of database coverage only", n, int(E7.sum()),
       ALL(lambda F: fit_gen(F, G1, T_=T7, E_=E7)), ALL(lambda F: fit_gen(F, Cl, T_=T7, E_=E7)))
print(f"[{(time.time()-t0)/60:.1f} min]")

In [ ]:
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter
ro.r("suppressMessages(library(survival))")

S_ = A.status.values
CENS = (S_ == 0) & (T < 730)
print(f"censored before 730 days: {CENS.sum():,}")

EDG = [0, 91, 182, 273, 364, 455, 546, 637, 730]
iv = []
for i, (t, s) in enumerate(zip(T, S_)):
    for k in range(8):
        a0, b0 = EDG[k], EDG[k + 1]
        if t <= a0: break
        last = t <= b0
        iv.append((i, k, a0, min(t, b0), int(last and s == 1), int(last and s == 0 and t < 730)))
L = pd.DataFrame(iv, columns=["i", "k", "start", "stop", "ev", "cens"])

clean = lambda c: re.sub(r"[^A-Za-z0-9]", "_", str(c))
CD = class_dummies(A).reset_index(drop=True); CD.columns = [clean(c) for c in CD.columns]
EXPO = pd.concat([CD, pd.DataFrame({"gap365": A.gap365.astype(float).values})], axis=1)
KD = pd.get_dummies(L.k, prefix="int", drop_first=True, dtype=float)

def cox_r(D, rhs, expo):
    with localconverter(ro.default_converter + pandas2ri.converter):
        ro.globalenv["d"] = ro.conversion.py2rpy(D)
    ro.r(f'm <- coxph(Surv(start, stop, ev) ~ {rhs}, data=d, weights=w, cluster=id, ties="efron")')
    cf = np.array(ro.r("coef(m)")); se = np.sqrt(np.diag(np.array(ro.r("vcov(m)"))))
    nm = list(ro.r("names(coef(m))"))
    return {e: (cf[nm.index(e)], se[nm.index(e)]) for e in expo}

EXP_SETS = {"class": list(CD.columns), "gap": ["gap365"]}
WTS = ["trunc 1/99", "trunc 5/95"]
cw = {(mo, w): [] for mo in EXP_SETS for w in WTS}
diag = {}
t0 = time.time()
for m in range(1, M + 1):
    F = imputed(m)
    X = build_X(F, KN).reset_index(drop=True); X.columns = [clean(c) for c in X.columns]
    XL = X.iloc[L.i.values].reset_index(drop=True); EL = EXPO.iloc[L.i.values].reset_index(drop=True)
    den = sm.GLM(L.cens, sm.add_constant(pd.concat([KD, XL, EL], axis=1)), family=sm.families.Binomial()).fit()
    num = sm.GLM(L.cens, sm.add_constant(pd.concat([KD, EL], axis=1)), family=sm.families.Binomial()).fit()
    ratio = pd.Series((1 - num.fittedvalues.values) / (1 - den.fittedvalues.values))
    w_raw = ratio.groupby(L.i).cumprod().groupby(L.i).shift(1).fillna(1.0).values
    W = {"trunc 1/99": np.clip(w_raw, *np.percentile(w_raw, [1, 99])),
         "trunc 5/95": np.clip(w_raw, *np.percentile(w_raw, [5, 95]))}
    if m == 1:
        ci = den.conf_int()
        ess = pd.DataFrame({"k": L.k, "w": W["trunc 1/99"]}).groupby("k").w.apply(lambda x: x.sum() ** 2 / (x ** 2).sum() / len(x) * 100)
        diag = {"weight_mean": float(w_raw.mean()), "weight_p1_p99": np.percentile(w_raw, [1, 99]).round(2).tolist(),
                "untruncated_gt5": int((w_raw > 5).sum()), "ess_pct_min": float(ess.min()),
                "censoring_OR": {e: [float(np.exp(den.params[e])), float(np.exp(ci.loc[e, 0])), float(np.exp(ci.loc[e, 1]))]
                                 for e in EL.columns}}
        print("imputation 1:", json.dumps(diag, indent=1))
    for mo, ex in EXP_SETS.items():
        D = pd.concat([L[["start", "stop", "ev"]], XL, EL[ex]], axis=1); D["id"] = L.i.values
        rhs = " + ".join(list(XL.columns) + ex)
        for wn in WTS:
            D["w"] = W[wn]; cw[(mo, wn)].append(cox_r(D, rhs, ex))
    if m % 5 == 0: print(f"  {m}/{M} [{(time.time()-t0)/60:.1f} min]")

def rubin(lst, e):
    q = np.array([d[e][0] for d in lst]); u = np.array([d[e][1] ** 2 for d in lst])
    Tv = u.mean() + (1 + 1 / len(q)) * q.var(ddof=1)
    return np.exp([q.mean(), q.mean() - 1.96 * np.sqrt(Tv), q.mean() + 1.96 * np.sqrt(Tv)])

sm_key = clean(SM)
for wn, lab in [("trunc 1/99", "Censoring weights, truncated at 1st/99th percentiles"),
                ("trunc 5/95", "Censoring weights, truncated at 5th/95th percentiles")]:
    g = rubin(cw[("gap", wn)], "gap365"); s_ = rubin(cw[("class", wn)], sm_key)
    rows.append({"analysis": lab, "n": n, "events": ev, "class_contrast": SM, "gap_HR": g[0], "gap_lo": g[1], "gap_hi": g[2],
                 "sm_HR": s_[0], "sm_lo": s_[1], "sm_hi": s_[2]})
    print(rows[-1])

tableS3 = pd.DataFrame(rows)

## 4 · Table S8 and Figure S3: outcome components and heart-failure checks (exploratory)

Myocardial infarction, stroke, and heart failure hospitalization as separate outcomes, by principal diagnosis and by any diagnosis position. Heart-failure checks add baseline heart failure and renal disease, add any heart-failure diagnosis before the landmark, or restrict to patients with neither.

In [ ]:
con = connect(MIMIC_HOSP, MIMIC_ED)
CODES = {"MI": (MI_9, MI_10), "Stroke": (STROKE_9, STROKE_10), "HF": (HF_9, HF_10)}
ids = A[["subject_id", "index_date"]].copy(); ids["index_date"] = pd.to_datetime(ids.index_date).dt.date
con.register("lm_ids", ids)
dx = con.execute('''SELECT d.subject_id, CAST(a.admittime AS DATE) - l.index_date AS day,
                           UPPER(TRIM(CAST(d.icd_code AS VARCHAR))) AS code, d.icd_version, d.seq_num
                    FROM diagnoses_icd d JOIN admissions a ON a.hadm_id = d.hadm_id
                    JOIN lm_ids l ON l.subject_id = d.subject_id''').df()
match = lambda p9, p10: ((dx.icd_version == 9) & dx.code.str.startswith(tuple(p9))) | \
                        ((dx.icd_version == 10) & dx.code.str.startswith(tuple(p10)))
comp = {k: match(*v) for k, v in CODES.items()}
comp["Composite"] = comp["MI"] | comp["Stroke"] | comp["HF"]
fu = A.fu_end.values

def outcome_for(rows_mask):
    hit = dx[rows_mask & (dx.day > 730)].groupby("subject_id").day.min()
    cday = A.subject_id.map(hit).values
    ev_ = ~np.isnan(cday) & (cday <= fu)
    dth = ~ev_ & A.death_day.notna().values & (A.death_day.values > 730) & (A.death_day.values <= fu)
    return (np.where(ev_, cday, np.where(dth, A.death_day.values, fu)) - 730).astype(float), ev_.astype(int)

t0 = time.time(); s8, fig_rows = [], []
for defn, pos in [("Principal diagnosis", dx.seq_num == 1), ("Any diagnosis position", pd.Series(True, index=dx.index))]:
    for oc in ["Composite", "MI", "Stroke", "HF"]:
        Tc, Ecomp = outcome_for(pos & comp[oc])
        tg = pool([fit_gen(imputed(m), G1, T_=Tc, E_=Ecomp) for m in range(1, M + 1)])[0]
        tc = pool([fit_gen(imputed(m), Cl, T_=Tc, E_=Ecomp) for m in range(1, M + 1)])[0]
        r = {"definition": defn, "outcome": oc, "events": int(Ecomp.sum()), "Gap >365 days": fmt_hr(tg, "gap365")}
        r.update({g: fmt_hr(tc, g) for g in ORDER[1:]}); s8.append(r)
        if defn.startswith("Principal"):
            fig_rows += [{"outcome": oc, "term": "Gap >365 days", **tg.loc["gap365", ["HR", "lo", "hi"]].to_dict()}] + \
                        [{"outcome": oc, "term": g, **tc.loc[g, ["HR", "lo", "hi"]].to_dict()} for g in ORDER[1:]]
        print(f"  {defn[:9]} {oc:9s} events {int(Ecomp.sum()):5d} [{(time.time()-t0)/60:.1f} min]")
tableS8a = pd.DataFrame(s8); figS3 = pd.DataFrame(fig_rows)
assert tableS8a.iloc[0].events == int(E1.sum()), "principal composite should equal the primary event count"

Thf, Ehf = outcome_for((dx.seq_num == 1) & comp["HF"])
cvb = cov.set_index("subject_id").loc[A.subject_id]
hf_pre = A.subject_id.isin(dx[comp["HF"] & (dx.day <= 730)].subject_id.unique()).astype(float).values
extra_base = pd.DataFrame({"chf_base": cvb.chf.fillna(0).values, "rf_base": cvb.rf.fillna(0).values})
neither = np.where((extra_base.chf_base == 0) & (hf_pre == 0))[0]
hf_rows = []
for name, ex, idx in [("Primary model", None, None),
                      ("+ baseline heart failure and renal disease", extra_base, None),
                      ("+ any heart-failure diagnosis before the landmark", pd.DataFrame({"hf_pre": hf_pre}), None),
                      ("No heart failure at baseline or before the landmark", None, neither)]:
    def one(m):
        F = imputed(m)
        if idx is None: return fit_gen(F, Cl, ex, T_=Thf, E_=Ehf)
        F = F.iloc[idx].reset_index(drop=True)
        return fit_gen(F, class_dummies(F), T_=Thf[idx], E_=Ehf[idx])
    t = pool([one(m) for m in range(1, M + 1)])[0]
    hf_rows.append({"model": name, "n": len(A) if idx is None else len(idx),
                    "events": int(Ehf.sum() if idx is None else Ehf[idx].sum()),
                    "Rising Contact": fmt_hr(t, "Rising Contact"), SM: fmt_hr(t, SM)})
tableS8b = pd.DataFrame(hf_rows)
pd.set_option("display.width", 250)
print(tableS8a.to_string(index=False)); print(tableS8b.to_string(index=False))

## 5 · Results summary and save

In [ ]:
fmt = lambda h, l, u: f"{h:.2f} ({l:.2f}-{u:.2f})"
show = tableS3.copy()
show["gap"] = [fmt(h, l, u) if pd.notna(h) else "" for h, l, u in zip(show.get("gap_HR"), show.get("gap_lo"), show.get("gap_hi"))]
show["class contrast"] = [fmt(h, l, u) if pd.notna(h) else "" for h, l, u in zip(show.get("sm_HR"), show.get("sm_lo"), show.get("sm_hi"))]
print(f"class contrast vs Minimal Contact: {SM} (other solution's row names its own contrast)")
print(show[["analysis", "n", "events", "gap", "class_contrast", "class contrast", "class_p"]].round(2).to_string(index=False))

print("\nTABLE S8A")
print(tableS8a.to_string(index=False))
print("\nTABLE S8B")
print(tableS8b.to_string(index=False))
print(f"\ncensoring weights (imputation 1): mean {diag['weight_mean']:.2f}, 1st-99th percentile {diag['weight_p1_p99']}, "
      f"untruncated weights > 5: {diag['untruncated_gt5']}, smallest effective sample size {diag['ess_pct_min']:.1f}% of patients at risk")
print("odds ratios for early censoring:", {k: [round(x, 2) for x in v] for k, v in diag["censoring_OR"].items()})

tableS3.to_csv(os.path.join(RESULTS_DIR, "tableS3.csv"), index=False)
tableS8a.to_csv(os.path.join(RESULTS_DIR, "tableS8a_components.csv"), index=False)
tableS8b.to_csv(os.path.join(RESULTS_DIR, "tableS8b_heart_failure.csv"), index=False)
figS3.to_csv(os.path.join(RESULTS_DIR, "figureS3.csv"), index=False)
save_json({"censoring_weights": diag}, RESULTS_DIR, "06b_sensitivity")